# Hydraulic Erosion

The `erode()` function simulates particle-based hydraulic erosion on a terrain raster. Thousands of virtual water droplets are dropped on the surface, each one tracing a path downhill. Along the way, droplets pick up sediment from steep slopes and deposit it when they slow down or encounter flatter ground.

This produces realistic valley networks and smoothed ridgelines from synthetic or real terrain data. The function supports numpy, cupy, dask+numpy, and dask+cupy backends. On GPU, each particle runs as a separate CUDA thread for large speedups on high-resolution grids.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial.erosion import erode

## 1. Generate synthetic terrain

We build a simple terrain using layered sine waves plus random noise. This gives enough slope variation for erosion to produce visible channels.

In [ ]:
size = 256
y = np.linspace(0, 4 * np.pi, size)
x = np.linspace(0, 4 * np.pi, size)
xx, yy = np.meshgrid(x, y)

# Layered sine terrain with noise
rng = np.random.default_rng(42)
terrain = (
    200 * np.sin(xx * 0.3) * np.cos(yy * 0.2)
    + 100 * np.sin(xx * 0.7 + yy * 0.5)
    + 50 * rng.random((size, size))
    + 500  # raise the baseline so values stay positive
)

agg = xr.DataArray(
    terrain.astype(np.float32),
    dims=['y', 'x'],
    attrs={'res': (1.0, 1.0)},
)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(agg.values, cmap='terrain')
ax.set_title('Original terrain')
ax.axis('off')
fig.colorbar(im, ax=ax, shrink=0.7, label='Elevation')
plt.tight_layout()
plt.show()

## 2. Basic erosion

Run erosion with default parameters and compare against the original.

In [ ]:
eroded = erode(agg, iterations=50000, seed=42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(agg.values, cmap='terrain')
axes[0].set_title('Before')
axes[0].axis('off')

axes[1].imshow(eroded.values, cmap='terrain')
axes[1].set_title('After erosion')
axes[1].axis('off')

diff = eroded.values - agg.values
vlim = max(abs(diff.min()), abs(diff.max()))
im = axes[2].imshow(diff, cmap='RdBu', vmin=-vlim, vmax=vlim)
axes[2].set_title('Elevation change')
axes[2].axis('off')
fig.colorbar(im, ax=axes[2], shrink=0.7, label='Change')

plt.tight_layout()
plt.show()

## 3. Parameter effects

The `params` dict controls the erosion behavior. The most impactful ones:

- **erosion**: how aggressively particles remove material (default 0.3)
- **capacity**: how much sediment water can carry (default 4.0)
- **deposition**: how quickly excess sediment is dropped (default 0.3)
- **radius**: size of the erosion brush in cells (default 3)

In [ ]:
configs = {
    'Default': None,
    'High erosion': {'erosion': 0.9},
    'Large brush': {'radius': 6},
    'High capacity': {'capacity': 12.0},
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (label, p) in zip(axes, configs.items()):
    result = erode(agg, iterations=30000, seed=42, params=p)
    diff = result.values - agg.values
    vlim = max(abs(diff.min()), abs(diff.max()), 1)
    ax.imshow(diff, cmap='RdBu', vmin=-vlim, vmax=vlim)
    ax.set_title(label)
    ax.axis('off')

plt.suptitle('Elevation change under different parameters', y=1.02)
plt.tight_layout()
plt.show()

## 4. Iterative refinement

More iterations means more droplets, which produces deeper, more developed channel networks.

In [ ]:
iter_counts = [5000, 20000, 50000, 100000]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, n in zip(axes, iter_counts):
    result = erode(agg, iterations=n, seed=42)
    ax.imshow(result.values, cmap='terrain')
    ax.set_title(f'{n:,} droplets')
    ax.axis('off')

plt.suptitle('Erosion depth vs. iteration count', y=1.02)
plt.tight_layout()
plt.show()

## 5. Cross-section comparison

A 1D slice through the terrain shows how erosion carves valleys and smooths peaks.

In [ ]:
row = size // 2
eroded_heavy = erode(agg, iterations=80000, seed=42)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(agg.values[row, :], label='Original', linewidth=2)
ax.plot(eroded.values[row, :], label='50k droplets', linewidth=1.5)
ax.plot(eroded_heavy.values[row, :], label='80k droplets', linewidth=1.5)
ax.set_xlabel('Column')
ax.set_ylabel('Elevation')
ax.set_title(f'Cross-section at row {row}')
ax.legend()
plt.tight_layout()
plt.show()